In [55]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [56]:
bronze_stream = (
    spark.readStream
    .format("delta")
    .load("/opt/spark-data/delta/bronze/orders_raw")
)
# 3. Define the Master Schema
payload_schema = StructType([
    StructField("store_details", StructType([
        StructField("store_id", StringType())
    ])),
    StructField("order", StructType([
        StructField("order_id", StringType()),
        StructField("order_time", StringType()),
        StructField("status", StringType()),
        StructField("order_events", ArrayType(StructType([
            StructField("event_type", StringType()),
            StructField("event_time", StringType())
        ]))),
        StructField("customer", StructType([
            StructField("customer_id", StringType())
        ])),
        StructField("charges", StructType([
            StructField("shipping_fee", DecimalType(10,2)),
            StructField("tax", DecimalType(10,2)),
            StructField("currency", StringType())
        ])),
        StructField("device_context", StructType([
            StructField("platform", StringType()),
            StructField("app_version", StringType())
        ])),
        StructField("delivery", StructType([
            StructField("delivery_partner", StructType([
                StructField("partner_id", StringType())
            ])),
            StructField("delivery_type", StringType()),
            StructField("dispatch_time", StringType()),
            StructField("actual_delivery_time", StringType()),
            StructField("expected_delivery_date", StringType()),
            StructField("sla_days", IntegerType()),
            StructField("delivery_attempts", IntegerType())
        ])),
        StructField("payments", ArrayType(StructType([
            StructField("payment_id", StringType()),
            StructField("method", StringType()),
            StructField("provider", StringType()),
            StructField("status", StringType()),
            StructField("failure_reason", StringType()),
            StructField("attempt_time", StringType())
        ]))),
        StructField("items", ArrayType(StructType([
            StructField("item_id", StringType()),
            StructField("product_details", StructType([
                StructField("product_id", StringType())
            ])),
            StructField("price", DecimalType(10,2)),
            StructField("quantity", IntegerType()),
            StructField("discounts", ArrayType(StructType([
                StructField("discount_type", StringType()),
                StructField("code", StringType()),
                StructField("amount", DecimalType(10,2))
            ])))
        ])))
    ]))
])
# 4. Create the Parsed DataFrame
parsed_df = bronze_stream.withColumn("data", from_json(col("raw_payload"), payload_schema))
print("Master DataFrame 'parsed_df' successfully created and listening to Bronze!")

Master DataFrame 'parsed_df' successfully created and listening to Bronze!
